# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR\u2072 mass spectrometry protein dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described using a Croissant schema, accessible via the URL below.

:point_right: **Schema URL:** https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset-level metadata
print(f"Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")
# Extra metadata can be accessed as shown below:
print(f"Published: {dataset.metadata.date_published}\nLicense: {dataset.metadata.license}\nKeywords: {getattr(dataset.metadata, 'keywords', None)}\n")

## 2. Data Overview
Let's examine available record sets, their fields, and entity `@id`s.

**Note:** All entities are referenced by their Croissant `@id` and `name` for clarity.

In [ ]:
def describe_record_sets(ds):
    record_sets = ds.record_sets
    print(f"Found {len(record_sets)} record set(s):\n")
    for rset in record_sets:
        print(f"- [@id] {rset.id} | [Name] {getattr(rset, 'name', None)}")
        if getattr(rset, 'fields', None):
            print("    Fields:")
            for f in rset.fields:
                print(f"        [@id] {f.id} | [Name] {getattr(f, 'name', None)} | [dataType] {getattr(f, 'data_type', None)}")
        else:
            print("    (No fields found)")
        print()

describe_record_sets(dataset)

### Show sample records from record set
Below, we display up to three sample records for each record set defined in the Croissant schema, referenced by their `@id`.

In [ ]:
for rset in dataset.record_sets:
    print(f"Record Set [@id]: {rset.id}  [Name]: {getattr(rset, 'name', None)}\n---")
    try:
        for idx, rec in enumerate(dataset.records(record_set=rset.id)):
            print(json.dumps(rec, indent=2))
            if idx >= 2:
                break
    except Exception as e:
        print(f"(Unable to load records for {rset.id}: {e})")
    print()

## 3. Data Extraction
Load all data for a chosen record set into a pandas DataFrame for further analysis and exploration.

We'll demonstrate extraction for the first record set. Use `record_sets_ids` as a list of their respective `@id`.

In [ ]:
# Get all record set @id's:
record_set_ids = [rset.id for rset in dataset.record_sets]
print(f"Record sets in dataset: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for {record_set_id} ({len(records)} records, {len(dataframes[record_set_id].columns)} columns)")
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# Pick one record set for demonstration (use actual @id from record sets, like the first one)
if record_set_ids:
    chosen_record_set = record_set_ids[0]
    print(f"\nFields in {chosen_record_set}: {list(dataframes[chosen_record_set].columns) if chosen_record_set in dataframes else 'No Data'}\n")
    if chosen_record_set in dataframes:
        dataframes[chosen_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field for analysis, filter, normalize, and group data.

> **Note:** Fields, columns, and grouping variables are referenced by `@id` from the schema. Please adapt this cell for the actual column names found in your chosen record set.

In [ ]:
# Replace with your actual record set @id from above and suitable numeric/grouping fields found in the data
record_set_id = chosen_record_set
df = dataframes.get(record_set_id, None)

if df is not None:
    # Pick a numeric field to demonstrate (find integer/float columns)
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric columns found: {numeric_cols}")

    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"\nAnalyzing numeric field: {numeric_field_id}\n")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}\n")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Sample normalized values:\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Pick a candidate group field (first string/object column with <50 unique values)
        grouping_candidates = [
            col for col in df.columns
            if df[col].dtype == object and df[col].nunique() < 50
        ]
        if grouping_candidates:
            group_field_id = grouping_candidates[0]
            print(f"\nGrouping by field: {group_field_id}\n")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric column found in the record set.")
else:
    print(f"No DataFrame found for record set {record_set_id}")

## 5. Visualization
Visualize distributions, such as a histogram of the selected numeric field and a boxplot grouped by a field.

_Modify field variables below according to your actual record set's field/column `@id` values._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_cols:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group field
    if 'group_field_id' in locals():
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Data not available for visualization.")

## 6. Conclusion
In this notebook, you used the `mlcroissant` library to access and explore the FAIR\u00b2 mass spectrometry protein dataset via its Croissant schema. 
* You listed available record sets and their fields by `@id`.
* You demonstrated loading data, filtering by a numeric field, normalization, grouping, and generating descriptive visualizations.

For deeper analysis, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api.html) and inspect additional record sets or fields referenced by `@id`.